# 03 · Modelos de potencia esperada

Compara cinco modelos (física con pvlib, Ridge, Random Forest, XGBoost y **híbrido físico + ML**, donde un
Random Forest aprende el factor real/físico) en dos escenarios de
irradiancia: **sensor de planta (LMD)**, que se usa tal cual porque ya mide en el plano del panel, y
**Open-Meteo**, que se pasa del plano horizontal al plano del panel con pvlib.

**Umbral de la hipótesis** (fijado en `docs/plan_modelado.md` antes de mirar mayo y junio):
error de energía diaria **< 8%** con el sensor de planta, en los meses reservados.

**Validación:** validación cruzada dejando un mes entero afuera en cada vuelta (jul-2018 a abr-2019).
Mayo y junio de 2019 se miran **una sola vez**, al final, con el modelo ya elegido.

**Fuera del entrenamiento:**
- **station04:** su ficha es imposible (25,7 MW con 20,06 MWp de módulos).
- **station09:** su bajo rendimiento real es el caso de prueba del detector (`04`).
- **Horas de station01 con datos congelados** (`limpiar_pvod`).
- **Apagones con sol** (más de 500 W/m² y menos del 5% de la capacidad).

**Lluvia y suciedad:** `dias_desde_lluvia` (días desde la última lluvia ≥ 5 mm, de la precipitación de
Open-Meteo) entra como variante "con lluvia". En `04`, lo esperado se calcula **con la planta limpia**
(`dias_desde_lluvia = 0`), así el residuo incluye la pérdida por suciedad.

> **Transparencia.** La primera versión de este notebook (commit `ab3630b`) comparó 4 modelos sin lluvia y
> miró mayo y junio **una sola vez**: Random Forest dio 6,04%. El híbrido y la variante con lluvia se
> agregaron **después**, por pedido del equipo. Sus resultados en mayo y junio ya no son una evaluación
> ciega. La condición 1 de la hipótesis se juzga con esa primera mirada; lo nuevo se informa aparte.

In [ ]:
# Si corrés en Colab: clonar el repo (una vez por sesión), pararse en él e instalar dependencias
import os, sys
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/tpo-solar'):
        !git clone https://github.com/imelinc/tpo-solar.git /content/tpo-solar
    %cd /content/tpo-solar
    !pip install -q -r requirements.txt
sys.path.append('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
from src.config import DATA_RAW, DATA_PROCESSED, MODELS

In [ ]:
import time, warnings
import joblib
from src.config import ROOT
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.modelos import (TIPOS, EXCLUIDAS, FEATURES, ModeloPotencia, preparar, filas_entrenamiento,
                         cv_por_mes, metricas, energia_diaria)

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.width", 200)
UMBRAL_HIPOTESIS = 8.0          # % de error de energía diaria (docs/plan_modelado.md)
CORTE_HOLDOUT = "2019-05-01"
A = pd.read_parquet(DATA_PROCESSED / "dataset_A_pvod_hourly.parquet")
D = {esc: preparar(A, esc) for esc in ["lmd", "om"]}

## 1. Datos de entrenamiento

In [ ]:
d = D["lmd"]
resumen = pd.DataFrame({
    "horas_sol": d.groupby("plant_id").size(),
    "congeladas": d.groupby("plant_id")["dato_congelado"].sum(),
    "apagones": d.groupby("plant_id")["apagon"].sum(),
    "entrena_dev": d[filas_entrenamiento(d) & (d["timestamp_local_utc8"] < CORTE_HOLDOUT)].groupby("plant_id").size(),
    "holdout": d[filas_entrenamiento(d) & (d["timestamp_local_utc8"] >= CORTE_HOLDOUT)].groupby("plant_id").size(),
}).fillna(0).astype(int)
print(resumen)
print("\nExcluidas del entrenamiento:", EXCLUIDAS)

## 2. Validación cruzada por mes (jul-2018 a abr-2019)

In [ ]:
filas, oof = [], {}
VARIANTES = [(esc, tipo, False) for esc in ["lmd", "om"] for tipo in TIPOS] + [
    ("lmd", tipo, True) for tipo in ["rf", "xgb", "hibrido"]]
for esc, tipo, lluvia in VARIANTES:
    dev = D[esc][filas_entrenamiento(D[esc]) & (D[esc]["timestamp_local_utc8"] < CORTE_HOLDOUT)]
    t0 = time.time()
    oof[(esc, tipo, lluvia)] = cv_por_mes(dev, tipo, esc, con_lluvia=lluvia)
    m = metricas(dev, oof[(esc, tipo, lluvia)])
    filas.append({"escenario": esc, "modelo": tipo, "con_lluvia": lluvia, **m, "seg_cv": time.time() - t0})
cv = pd.DataFrame(filas)
print(cv.round(3).to_string(index=False))

**Conclusión:** Con el sensor de planta, el mejor es el **híbrido físico + ML**: 5,99% de error de energía diaria (R² de 0,968). Le siguen XGBoost (6,40%) y Random Forest (6,51%); Ridge queda en 8,0% y la física sola en 13,6%. El híbrido funciona mejor porque reparte el trabajo: la física da la forma (irradiancia y temperatura) y el Random Forest solo corrige el factor (nivel y estacionalidad de cada planta), así que tiene menos para aprender. Con Open-Meteo todos empeoran mucho (16-22%). La variante con días desde lluvia **no mejora la validación cruzada**: el híbrido da 6,05% con lluvia contra 5,99% sin ella, y Random Forest 6,65% contra 6,51%. Con un solo año, la época seca cae en invierno y el efecto de la suciedad se confunde con la estación. Ningún modelo baja del 5% original.

### ⭐ Error de energía diaria vs. umbral de la hipótesis

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
nombres = {"fisico": "Física (pvlib)", "ridge": "Ridge", "rf": "Random Forest", "xgb": "XGBoost", "hibrido": "Físico + RF"}
esc_lbl = {"lmd": "Sensor de planta", "om": "Open-Meteo (satélite)"}
g = cv[~cv["con_lluvia"]].assign(Modelo=cv["modelo"].map(nombres), Escenario=cv["escenario"].map(esc_lbl))
sns.barplot(data=g, x="Modelo", y="error_energia_diaria_pct", hue="Escenario", ax=ax, palette=["#3182bd", "#fdae6b"])
ax.axhline(UMBRAL_HIPOTESIS, color="red", ls="--", label=f"umbral {UMBRAL_HIPOTESIS:.0f}%")
for p in ax.patches:
    if p.get_height() > 0:
        ax.annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom", fontsize=8)
ax.set(ylabel="error de energía diaria (%)", xlabel="", title="Validación cruzada por mes: cuánto se equivoca cada modelo en la energía del día")
ax.legend(); plt.show()

### Error por planta y mes (mejores modelos con sensor de planta)

In [ ]:
dev = D["lmd"][filas_entrenamiento(D["lmd"]) & (D["lmd"]["timestamp_local_utc8"] < CORTE_HOLDOUT)]
e = energia_diaria(dev, oof[("lmd", "rf", False)]).assign(mes=lambda x: pd.to_datetime(x["date_local"]).dt.month)
e["abs"] = (e["e_real"] - e["e_esp"]).abs()
tabla = (100 * e.groupby(["plant_id", "mes"])["abs"].sum() / e.groupby(["plant_id", "mes"])["e_real"].sum()).unstack()
sesgo = (100 * (e.groupby(["plant_id", "mes"])["e_real"].sum() - e.groupby(["plant_id", "mes"])["e_esp"].sum())
         / e.groupby(["plant_id", "mes"])["e_real"].sum()).unstack()
fig, axs = plt.subplots(1, 2, figsize=(15, 4))
sns.heatmap(tabla, annot=True, fmt=".0f", cmap="Reds", ax=axs[0], cbar_kws={"label": "%"})
axs[0].set(title="Random Forest: error de energía diaria (%)", xlabel="mes", ylabel="")
sns.heatmap(sesgo, annot=True, fmt=".0f", cmap="RdBu", center=0, ax=axs[1], cbar_kws={"label": "%"})
axs[1].set(title="Sesgo mensual (real − esperado, %): negativo = genera menos de lo esperado", xlabel="mes", ylabel="")
plt.tight_layout(); plt.show()
print("Sesgo mensual de station00 y station06 en invierno (nov-feb):")
print(sesgo.loc[["station00", "station06"], [11, 12, 1, 2]].round(1))

**Conclusión:** La mayoría de las plantas y meses quedan entre 3% y 9% de error. La excepción es **station06 en invierno**: en febrero el modelo espera un 35% más de lo que genera y en enero un 15% más. Es una limitación de los datos, no del modelo. Con un solo año, al dejar febrero afuera el modelo nunca vio la pérdida invernal de esa planta, así que no la puede aprender. station00 muestra lo mismo en diciembre, en menor medida (−11%). Con varios años, la planta como variable más el día del año deberían absorberlo. Con estos datos, station06 genera falsas alarmas en invierno (se confirma en `04`).

## 3. Elección del modelo

In [ ]:
lmd = cv[cv["escenario"] == "lmd"].set_index(["modelo", "con_lluvia"])
orden = lmd["error_energia_diaria_pct"].sort_values()
print("Error de energía diaria en CV (sensor de planta):"); print(orden.round(2))
# Regla fijada antes del holdout: si la diferencia es < 0,5 pp, gana el más simple de desplegar
# (sin dependencias extra; XGBoost en Mac necesita libomp para cargar en el dashboard offline).
# A igual tipo se prefiere CON lluvia: es la única variante que permite calcular lo esperado con la
# planta limpia (dias_desde_lluvia = 0), que es lo que usa el detector para que el residuo sea pérdida.
SIMPLICIDAD = {"fisico": 0, "ridge": 1, "rf": 2, "hibrido": 3, "xgb": 4}
empatados = orden[orden - orden.iloc[0] < 0.5].index
ELEGIDO, LLUVIA = min(empatados, key=lambda k: (SIMPLICIDAD[k[0]], not k[1]))
print("A menos de 0,5 pp del mejor:", list(empatados), "-> elegido:", ELEGIDO, "| con lluvia:", LLUVIA)

In [ ]:
practico = pd.DataFrame({
    "interpretable": {"fisico": "sí (1 parámetro por planta)", "ridge": "sí (coeficientes)", "rf": "parcial (importancia de variables)",
                      "xgb": "parcial (importancia de variables)", "hibrido": "parcial (física + factor de corrección)"},
    "dependencias": {"fisico": "pvlib", "ridge": "scikit-learn", "rf": "scikit-learn", "xgb": "xgboost (+ libomp en Mac)",
                     "hibrido": "pvlib + scikit-learn"},
})
base = cv[~cv["con_lluvia"]]
tabla_final = (base.pivot(index="modelo", columns="escenario", values="error_energia_diaria_pct").add_prefix("err_diario_%_")
               .join(cv[cv["con_lluvia"]].set_index("modelo")["error_energia_diaria_pct"].rename("err_diario_%_lmd_con_lluvia"))
               .join(base[base["escenario"] == "lmd"].set_index("modelo")[["MAE", "RMSE", "R2", "sesgo_energia_pct", "seg_cv"]].add_suffix("_lmd"))
               .join(practico))
print(tabla_final.loc[TIPOS].round(3).to_string())

**Conclusión:** Tres opciones quedan a menos de 0,5 puntos de la mejor: el híbrido sin lluvia (5,99%), el híbrido con lluvia (6,05%) y XGBoost (6,40%). Por la regla, gana el tipo más simple de desplegar: el **híbrido** (pvlib + scikit-learn, sin libomp). Entre sus dos variantes, la diferencia es de 0,05 puntos (ruido), y se elige **con lluvia** porque es la única que permite calcular lo esperado con la planta limpia, que el detector necesita para que el residuo sea pérdida. **Transparencia:** esta preferencia se fijó en la misma corrida en la que ya se veían los resultados de mayo y junio de las variantes nuevas. El motivo es funcional, pero no es una elección ciega.

**Cuándo conviene cada modelo:**
- **Física (pvlib):** cuando hay pocos datos o se necesita explicarlo en una línea. Es el piso de referencia y sirve para plantas nuevas sin historia.
- **Ridge:** cuando importa la interpretabilidad y la velocidad. Captura bien lo lineal en la irradiancia, pero no los recortes ni las sombras.
- **Random Forest:** buen rendimiento sin calibrar casi nada y sin dependencias extra. Conviene cuando se busca precisión con robustez y facilidad de despliegue.
- **XGBoost:** cuando se busca la máxima precisión y se puede invertir en calibrar hiperparámetros. Agrega una dependencia nativa (libomp en Mac).
- **Físico + ML (híbrido):** cuando hay poca historia o cambian las condiciones: la física aporta la forma y el ML solo corrige el factor (sombras, recortes, suciedad), así extrapola mejor fuera de lo visto.

## 4. Meses reservados (mayo y junio de 2019)

Los 4 modelos sin lluvia ya se miraron una vez, a ciegas (commit `ab3630b`). El híbrido y las variantes con lluvia se agregaron después y **no son una evaluación ciega**.

In [ ]:
hold_filas, PRED_HOLD = [], {}
for esc, tipo, lluvia in VARIANTES:
    d = D[esc]; ok = filas_entrenamiento(d)
    dev, hold = d[ok & (d["timestamp_local_utc8"] < CORTE_HOLDOUT)], d[ok & (d["timestamp_local_utc8"] >= CORTE_HOLDOUT)]
    PRED_HOLD[(esc, tipo, lluvia)] = ModeloPotencia(tipo, esc, con_lluvia=lluvia).fit(dev).predict(hold)
    ciega = (tipo in ("fisico", "ridge", "rf", "xgb")) and not lluvia
    hold_filas.append({"escenario": esc, "modelo": tipo, "con_lluvia": lluvia, "evaluacion_ciega": ciega,
                       **metricas(hold, PRED_HOLD[(esc, tipo, lluvia)])})
holdout = pd.DataFrame(hold_filas)
print(holdout.round(3).to_string(index=False))
res0 = holdout.set_index(["escenario", "modelo", "con_lluvia"]).loc[("lmd", "rf", False)]
print(f"\nCondición 1 (evaluación ciega, Random Forest sin lluvia): {res0['error_energia_diaria_pct']:.2f}% "
      f"-> umbral {UMBRAL_HIPOTESIS:.0f}%: {'CUMPLE' if res0['error_energia_diaria_pct'] < UMBRAL_HIPOTESIS else 'NO CUMPLE'}")
res = holdout.set_index(["escenario", "modelo", "con_lluvia"]).loc[("lmd", ELEGIDO, LLUVIA)]
print(f"Modelo elegido ahora ({ELEGIDO}, con lluvia={LLUVIA}): {res['error_energia_diaria_pct']:.2f}%")

**Conclusión:** **Condición 1, evaluación ciega** (la primera mirada, con Random Forest sin lluvia): **6,04%, por debajo del umbral de 8%. Se cumple.** Las variantes nuevas, ya sin ceguera, dan: híbrido con lluvia 5,04%, híbrido sin lluvia 5,59%, XGBoost con lluvia 5,73% y Random Forest con lluvia 5,86%. Con lluvia mejoran en mayo y junio (empieza la temporada de lluvias), pero eso no apareció en la validación cruzada, así que no se puede afirmar que generalice. Con Open-Meteo, el mejor da 10,4% (híbrido): sigue lejos de servir para detectar caídas de 10% día a día.

### Real vs. esperado en los meses reservados

In [ ]:
d = D["lmd"]; ok = filas_entrenamiento(d)
hold = d[ok & (d["timestamp_local_utc8"] >= CORTE_HOLDOUT)].assign(esperado=PRED_HOLD[("lmd", ELEGIDO, LLUVIA)])
eh = energia_diaria(hold, hold["esperado"])
fig, axs = plt.subplots(1, 2, figsize=(15, 4), gridspec_kw={"width_ratios": [2, 1]})
sem = hold[(hold["plant_id"] == "station08") & hold["timestamp_local_utc8"].between("2019-05-20", "2019-05-27")]
axs[0].plot(sem["timestamp_local_utc8"], sem["pvod_power_norm"], ".-", label="real")
axs[0].plot(sem["timestamp_local_utc8"], sem["esperado"], ".-", label="esperado")
axs[0].set(title="station08, una semana de mayo (horas de sol)", ylabel="potencia / capacidad"); axs[0].legend()
axs[1].scatter(eh["e_esp"], eh["e_real"], s=8, alpha=0.6)
lim = [0, eh[["e_esp", "e_real"]].max().max()]
axs[1].plot(lim, lim, "k--", lw=1)
axs[1].set(xlabel="energía esperada (MWh/MW/día)", ylabel="energía real", title="Energía diaria, todas las plantas")
plt.tight_layout(); plt.show()

## 5. Modelo final

Se reentrena con todos los datos limpios (dev + meses reservados). También se guarda una versión **planta típica** (sin la planta como variable) para plantas sin historia en el entrenamiento, como station09.

In [ ]:
from datetime import datetime
from importlib.metadata import version
ESC = "lmd"
d = D[ESC]; ok = filas_entrenamiento(d)
final = ModeloPotencia(ELEGIDO, ESC, con_lluvia=LLUVIA).fit(d[ok])
generico = ModeloPotencia(ELEGIDO, ESC, con_planta=False, con_lluvia=LLUVIA).fit(d[ok])
req = [l.split("==")[0] for l in (ROOT / "requirements.txt").read_text().split() if "==" in l]
meta = {
    "modelo": final, "modelo_generico": generico, "tipo": ELEGIDO, "escenario": ESC, "con_lluvia": LLUVIA,
    "fecha": datetime.now().isoformat(timespec="seconds"), "features": final.features,
    "target": "pvod_power_norm (potencia / capacidad), horas de sol",
    "plantas_entrenamiento": final.plantas_, "excluidas": {
        "station04": "ficha imposible: 25,7 MW con 20,06 MWp de módulos",
        "station09": "bajo rendimiento real, caso de prueba del detector"},
    "metricas_cv": cv.set_index(["escenario", "modelo", "con_lluvia"]).loc[(ESC, ELEGIDO, LLUVIA)].to_dict(),
    "metricas_holdout": res.to_dict(),
    "metricas_holdout_ciega_rf": res0.to_dict(),
    "umbral_hipotesis_pct": UMBRAL_HIPOTESIS,
    "versiones": {p: version(p) for p in req},
}
MODELS.mkdir(exist_ok=True)
joblib.dump(meta, MODELS / "modelo_potencia.pkl", compress=3)
print("Guardado:", MODELS / "modelo_potencia.pkl", f"({(MODELS / 'modelo_potencia.pkl').stat().st_size / 1e6:.1f} MB)")
print({k: v for k, v in meta.items() if k not in ("modelo", "modelo_generico")})

In [ ]:
imp = pd.Series(final.est_.feature_importances_, index=final.columnas_).sort_values(ascending=False) if ELEGIDO in ("rf", "xgb", "hibrido") else None
if imp is not None:
    print("Importancia de variables (top 10):"); print(imp.head(10).round(3))

**Conclusión:** Modelo final: **híbrido con lluvia**, reentrenado con todos los datos limpios. Se guardó con sus metadatos y con la versión planta típica. En la importancia de variables (del factor de corrección) pesan el día del año (doy_cos 0,17), la planta (station01 0,13 y station06 0,10), la fracción difusa y el índice de cielo claro; los días desde lluvia pesan 0,043. O sea, el factor aprende sobre todo el nivel y la estacionalidad de cada planta, que es justo lo que la física sola no ve. El archivo pesa unos 43 MB (dos bosques), así que no se sube a git: se regenera corriendo este notebook en un minuto y medio.